## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Inception v2

__Probado en:__ tensorflow 2.10.1
*****

Basada en las implementaciones: [Anas BRITAL](https://medium.com/@AnasBrital98/googlenet-cnn-architecture-explained-inception-v1-225ae02513fd) - [Nitish Kumar Pilla](https://nitishkumarpilla.medium.com/understand-googlenet-inception-v1-and-implement-it-easily-from-scratch-using-tensorflow-and-keras-5404239f361)


## Librerias

In [ ]:
import matplotlib.pyplot as plt
import cv2 
from numpy import array, arange, ceil, random
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.utils import Sequence, to_categorical

## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model
  
  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics 
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

In [ ]:
class ImageArrayGenerator(Sequence):
    def __init__(self, images, labels=None, img_rows=224, img_cols=224, batch_size=32, shuffle=True):
        """
            Generador personalizada
        """
        self.images = images
        self.labels = labels
        self.img_rows = img_rows
        self.img_cols = img_cols
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = arange(len(images))
        self.on_epoch_end()

        print(f'Generator (size): {len(images)}')

    def __len__(self):
        # Número de batches por época
        return int(ceil(len(self.images) / self.batch_size))

    def __getitem__(self, index):
        # Genera un batch de datos
        batch_indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        batch_images = array([cv2.resize(img, (self.img_rows, self.img_cols)) for img in self.images[batch_indexes]]) / 255.0 

        if self.labels is not None:
            batch_labels = to_categorical(self.labels[batch_indexes])
            return batch_images, (batch_labels, batch_labels, batch_labels)
        else:
            return batch_images

    def on_epoch_end(self):
        # Mezcla los índices después de cada época
        if self.shuffle:
            random.shuffle(self.indexes)


## Diseño del modelo

In [ ]:
def initBlock(x):
    """
        INPUT:
            @param x: feature tensor
            @type x: tensor

        OUTPUT:
            @param x: feature tensor
            @type x: tensor
    """
    x = layers.Conv2D(filters = 64, kernel_size = (7,7), strides=2 , 
               padding='valid' , activation='relu',
               name='InitBlock_Conv2D_01')(x)
    x = layers.MaxPool2D(pool_size=(3,3) , strides=2,
                         name='InitBlock_MaxPool_01')(x)
    x = layers.Conv2D(filters = 64, kernel_size = (1,1), strides=1 , 
                      padding='same' , activation='relu',
                      name='InitBlock_Conv2D_02')(x)
    x = layers.Conv2D(filters = 192, kernel_size = (3,3), strides=1 , 
                      padding='same' , activation='relu',
                      name='InitBlock_Conv2D_03')(x)
    x = layers.MaxPool2D(pool_size=(3,3) , strides=2,
                         name='InitBlock__MaxPool_02')(x)
    return x

def InceptionBlock(x , nbr_f1 , nbr_f2_1 , nbr_f2_2 , nbr_f3_1 , nbr_f4, block_id=0) :
    """
        INPUT:
            @param x: feature tensor
            @type x: tensor

            @param nbr_f1: number of filters
            @type nbr_f1: int

            @param nbr_f2_1: number of filters
            @type nbr_f2_1: int

            @param nbr_f2_2: number of filters
            @type nbr_f2_2: int

            @param nbr_f3_1: number of filters
            @type nbr_f3_1: int

            @param nbr_f4: number of filters
            @type nbr_f4: int

            @param block_id: Inception block id
            @type block_id: int 

        OUTPUT:
            @param output_Layer: Inception Block feature tensor
            @type output_Layer: tensor
    """
    
    #Path 1
    path1 = layers.Conv2D(filters=nbr_f1, kernel_size = (1,1), 
                          padding='same' , activation='relu', 
                          name=f'InceptionBlock_{block_id}_P1_Conv2D')(x)
    
    #Path 2 
    path2 = layers.Conv2D(filters=nbr_f2_1, kernel_size = (1,1), 
                          padding='same' , activation='relu',
                          name=f'InceptionBlock_{block_id}_P2_Conv2D_01')(x)
    path2 = layers.Conv2D(filters=nbr_f2_2, kernel_size = (3,3), 
                          padding='same' , activation='relu',
                          name=f'InceptionBlock_{block_id}_P2_Conv2D_02')(path2)
    
    #Path 3
    path3 = layers.Conv2D(filters=nbr_f3_1, kernel_size = (1,1), 
                          padding='same' , activation='relu',
                          name=f'InceptionBlock_{block_id}_P3_Conv2D_01')(x)
    path3 = layers.Conv2D(filters=nbr_f3_1, kernel_size = (5,5), 
                          padding='same' , activation='relu',
                          name=f'InceptionBlock_{block_id}_P3_Conv2D_02')(path3)
    
    #Path 4
    path4 = layers.MaxPool2D(pool_size=(3,3) , strides=(1,1) , padding='same',
                             name=f'InceptionBlock_{block_id}_P4_MaxPool') (x)
    path4 = layers.Conv2D(filters=nbr_f4, kernel_size = (1,1), 
                          padding='same' , activation='relu',
                          name=f'InceptionBlock_{block_id}_P4_Conv2D')(path4)
    
    ## Concatenate paths
    output_Layer = layers.Concatenate(axis=-1, name=f'InceptionBlock_{block_id}_Concat')([path1 , path2 , path3 , path4])
    
    return output_Layer

def Auxiliary_classifier(x, output_units=10, block_id=0):
    """
        INPUT:
            @param x: feature tensor
            @type x: tensor

        OUTPUT:
            @param x: feature tensor
            @type x: tensor
    """

    x = layers.AveragePooling2D(pool_size = (5,5), strides = 3,
                                name=f'AC_{block_id}_AveragePool')(x)
    x = layers.Conv2D(filters = 128, kernel_size = (1,1), 
                      padding = 'same', activation = 'relu',
                      name=f'AC_{block_id}_Conv2D')(x)
    x = layers.Flatten(name=f'AC_{block_id}_Flatten')(x)
    x = layers.Dense(1024, activation = 'relu',
                     name=f'AC_{block_id}_Dense_01')(x)
    x = layers.Dropout(rate=0.7,
                       name=f'AC_{block_id}_Dropout')(x)
    x = layers.Dense(output_units, activation = 'softmax',
                     name=f'AC_{block_id}_output')(x)
    return x

def InceptionV1(input_shape=(224, 224, 3), output_units=10):
    """
        INPUT:
            @param input_shape: input shape (default: (224, 224, 3))
            @type input_shape: tensor

            @param output_units: number of units of the model output layer (default: 10). 
            @type output_units: int
            
        OUTPUT:
            @param model: Inception v1 model
            @type model: tensorflow.keras.Model
    """
    input_layer = layers.Input(shape = input_shape, 
                               name='Input')
    
    x1 = initBlock(x=input_layer)
    
    x1 = InceptionBlock(x=x1, nbr_f1=64, nbr_f2_1=96, nbr_f2_2=128, nbr_f3_1=16, nbr_f4=32, 
                        block_id=0)
    x1 = InceptionBlock(x=x1, nbr_f1=128, nbr_f2_1=128, nbr_f2_2=192, nbr_f3_1=32, nbr_f4=64, 
                        block_id=1)
    x1 = layers.MaxPool2D(pool_size=(3,3) , strides=2, 
                          name='Incept_MaxPool_01')(x1)
    x1 = InceptionBlock(x=x1, nbr_f1=192, nbr_f2_1=96, nbr_f2_2=208, nbr_f3_1=16, nbr_f4=64, 
                        block_id=2)
    
    # Auxiliary classifier 1
    x2 = Auxiliary_classifier(x=x1, output_units=10, block_id=0)

    x1 = InceptionBlock(x=x1, nbr_f1=160, nbr_f2_1=112, nbr_f2_2=224, nbr_f3_1=24, nbr_f4=64, 
                        block_id=3)
    x1 = InceptionBlock(x=x1, nbr_f1=128, nbr_f2_1=128, nbr_f2_2=256, nbr_f3_1=24, nbr_f4=64, 
                        block_id=4)
    x1 = InceptionBlock(x=x1, nbr_f1=112, nbr_f2_1=144, nbr_f2_2=288, nbr_f3_1=32, nbr_f4=64, 
                        block_id=5)
    
    # Auxiliary classifier 2
    x3 = Auxiliary_classifier(x=x1, output_units=10, block_id=1)
    
    x1 = InceptionBlock(x=x1, nbr_f1=256, nbr_f2_1=160, nbr_f2_2=320, nbr_f3_1=32, nbr_f4=128, 
                        block_id=6)
    x1 = layers.MaxPool2D(pool_size=(3,3), strides=2, 
                          name='Incept_MaxPool_02')(x1)
    x1 = InceptionBlock(x=x1, nbr_f1=256, nbr_f2_1=160, nbr_f2_2=320, nbr_f3_1=32, nbr_f4=128, 
                        block_id=7)
    x1 = InceptionBlock(x=x1, nbr_f1=384, nbr_f2_1=192, nbr_f2_2=384, nbr_f3_1=48, nbr_f4=128, 
                        block_id=8)
    
    x1 = layers.GlobalAveragePooling2D(name = 'Incept_GAPool')(x1)
    x1 = layers.Dropout(rate=0.4, 
                        name='Incept_Dropout')(x1)
    x1 = layers.Dense(units=1000, activation='relu', 
                      name='Incept_Dense')(x1)
    x1 = layers.Dense(units=output_units, activation='softmax',
                      name='Incept_output')(x1)

    model = Model(input_layer, [x1 , x2 , x3] , name='InceptionV1')
    return model

## Dataset

<center>
    <img src=https://docs.pytorch.org/tutorials/_images/cifar10.png width=800>
</center>

El conjunto de datos CIFAR-10 consta de 60000 imágenes en color de 32x32 en 10 clases, con 6000 imágenes por clase. Hay 50000 imágenes de entrenamiento y 10000 imágenes de prueba.

Las etiquetas están codificadas de la siguiente forma:

| Label | Description |
|-------|-------------|
| 0     | airplane    |
| 1     | automobile  |
| 2     | bird        |
| 3     | cat         |
| 4     | deer        |
| 5     | dog         |
| 6     | frog        |
| 7     | horse       |
| 8     | ship        |
| 9     | truck       |


**Objetivo**: Clasificar las imágenes según su clase. 


#### Carga de datos

In [ ]:
#X_train, y_train, X_test, y_test = cifar10_data(img_rows=224, img_cols=224, max_sample=None)
(X_trainVal, y_trainVal), (X_test, y_test) = cifar10.load_data()

X_train, X_val, y_train, y_val = train_test_split(X_trainVal, y_trainVal, stratify=y_trainVal, 
                                                  test_size=0.1)

print('Train (shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('Validation (shape) X: {}, y: {}'.format(X_val.shape, y_val.shape))
print('Test (shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))

In [ ]:
## Instancias de generadores
train_generator = ImageArrayGenerator(X_train, y_train, batch_size=256, shuffle=True)
val_generator = ImageArrayGenerator(X_val, y_val, batch_size=256, shuffle=False)
test_generator = ImageArrayGenerator(X_test, y_test, batch_size=256, shuffle=False)

## Modelo

In [ ]:
model = InceptionV1(input_shape=(224, 224, 3), output_units=10)
model.summary()

#### Ajuste del modelo

In [ ]:
## Tiempo estimado de ejecución: 42 minutos (GPU)

## Configuración del modelo
model.compile(loss=['categorical_crossentropy', 'categorical_crossentropy', 'categorical_crossentropy'], 
              loss_weights=[1, 0.3, 0.3], optimizer='adam', metrics=['accuracy'])

## Ajuste del modelo
history = model.fit(train_generator, validation_data=val_generator, epochs=50)

In [ ]:
## Grafica de desempeño
plot_history(history, width=12, height=12)

## Computo de predicción

In [ ]:
## Predicciones
predicciones = model.predict(test_generator)

## Selección de salida del modelo
prediccion_labels = predicciones[0].argmax(axis=1)

## Etiqutas reales
y_labels = test_generator.labels.flatten()

In [ ]:
## Display confusion matrix
cm = confusion_matrix(y_pred=prediccion_labels, y_true=y_labels)
CM = ConfusionMatrixDisplay(confusion_matrix=cm)
CM.plot()
plt.show()

In [ ]:
## Display classification report
print(classification_report(y_pred=prediccion_labels, 
                            y_true=y_labels, 
                            digits=2) )